# ⚖️ Chapter 08: In-Database Huff Gravity Model & Market Catchments
## Modeling Spatial Economics and Retail Patronage Probabilities

### 🎯 The Mission: Why We Are Doing This
Spatial planners in Accra need to understand consumer catchment probabilities for major commercial hubs (**Makola**, **Kaneshie**, **Madina**). We implement the **Huff Gravity Formulation** directly in Spatial SQL using window functions:
$$P_{ij} = \frac{S_j \cdot d_{ij}^{-\lambda}}{\sum_k S_k \cdot d_{ik}^{-\lambda}}$$

---

### 🔑 Spatial DBA Keywords Decoded

| Term | Plain-English Explanation | Production Value |
| :--- | :--- | :--- |
| **Huff Model ($P_{ij}$)** | Gravitational interaction formula predicting consumer choice based on floor area ($S$) and distance ($d$). | Retail site selection & trade area delineation. |
| **Friction Exponent ($\lambda$)** | Travel impedance factor (typically $1.8$ to $2.0$). | Models consumer reluctance to travel far. |
| **`PARTITION BY hex_id`** | Normalizes raw gravity scores so probabilities sum to $1.0$ per cell. | In-database spatial econometric modeling. |

In [ ]:
from config_db import execute_sql, query_df, query_gdf, preview_map

sql_huff = """
DROP TABLE IF EXISTS central_markets CASCADE;
CREATE TABLE central_markets (
    market_id SERIAL PRIMARY KEY,
    name VARCHAR(100),
    attractiveness_m2 INT,
    geom GEOMETRY(Point, 32630)
);

INSERT INTO central_markets (name, attractiveness_m2, geom)
VALUES 
    ('Makola Central Market', 60000, ST_SetSRID(ST_MakePoint(809200, 612400), 32630)),
    ('Kaneshie Market Complex', 35000, ST_SetSRID(ST_MakePoint(806100, 614500), 32630)),
    ('Madina Zongo Market', 40000, ST_SetSRID(ST_MakePoint(814500, 627100), 32630));

DROP TABLE IF EXISTS huff_market_probabilities CASCADE;
CREATE TABLE huff_market_probabilities AS
WITH od_matrix AS (
    SELECT 
        h.hex_id, ST_Centroid(h.geom) AS origin_geom,
        m.market_id, m.name AS market_name, m.attractiveness_m2,
        ST_Distance(ST_Centroid(h.geom), m.geom) AS distance_m
    FROM accra_hex_grid_250m h
    CROSS JOIN central_markets m
),
gravity_scores AS (
    SELECT 
        hex_id, origin_geom, market_id, market_name,
        attractiveness_m2 / POWER(NULLIF(distance_m / 1000.0, 0), 1.8) AS raw_score
    FROM od_matrix
),
probabilities AS (
    SELECT 
        hex_id, origin_geom, market_id, market_name,
        raw_score / NULLIF(SUM(raw_score) OVER(PARTITION BY hex_id), 0) AS prob_patronage
    FROM gravity_scores
)
SELECT hex_id, origin_geom, market_id, market_name, ROUND(prob_patronage::NUMERIC, 4) AS prob_patronage
FROM probabilities;
"""

execute_sql(sql_huff)
print("✅ Huff Gravity model evaluated in SQL.")

In [ ]:
# View Dominant Catchment Shares
df_huff = query_df("""
SELECT market_name, COUNT(*) AS dominant_cell_count, ROUND(AVG(prob_patronage)::NUMERIC, 3) AS avg_probability
FROM huff_market_probabilities
WHERE prob_patronage > 0.50
GROUP BY market_name;
""")

df_huff

In [ ]:
# Interactive Map Preview: Market Trade Areas
gdf_huff = query_gdf("""
SELECT hex_id, market_name, prob_patronage, origin_geom AS geom
FROM huff_market_probabilities
WHERE prob_patronage > 0.50;
""")

preview_map(gdf_huff, tooltip_cols=["market_name", "prob_patronage"])